# CreditPilot — Dataset Generation

## Overview
This notebook focuses on generating a **realistic financial dataset** for building an AI-driven credit risk assessment system. The dataset simulates real-world lending scenarios by incorporating financial, behavioral, and credit-related attributes of loan applicants.

---

## Objective
The main objectives of this notebook are:

- Generate a **high-quality synthetic dataset (10,000+ records)**  
- Ensure **realistic relationships** between financial variables  
- Create a robust **target variable (`loan_status`)** using risk-based logic  
- Prepare data for **machine learning model training and analysis**

---

## Dataset Features

### Applicant Profile
- Age  
- Employment type  
- Employment length  
- Annual income  
- Income stability score  

### Loan Details
- Loan type  
- Loan amount  
- Loan tenure  
- Interest rate  
- EMI (calculated using standard financial formula)  

### Financial Health Metrics
- Debt-to-Income Ratio (DTI)  
- EMI-to-Income Ratio  
- Loan-to-Income Ratio  
- Credit utilization  

### Behavioral Features
- Number of credit inquiries  
- Late payment count  
- Default history  
- Repayment history score  
- Job stability score  

---

## Target Variable: Loan Status

The `loan_status` variable represents whether a loan is:

- **0 → Safe (No default)**  
- **1 → Risky (Default likely)**  

It is generated using a **multi-factor risk scoring system** based on:

- Credit score  
- Financial burden (DTI, EMI ratio)  
- Credit utilization  
- Behavioral indicators (late payments, inquiries)  

A small amount of randomness is introduced to simulate **real-world uncertainty**.

---

## Methodology

1. Generate base features using statistical distributions  
2. Create dependencies between financial variables  
3. Compute derived features (EMI, ratios, etc.)  
4. Build a **risk score → probability → classification** pipeline  
5. Generate the final dataset  

---

## Output

- File: `creditpilot_dataset.csv`  
- Records: **10,000 rows**  
- Features: **25+ columns**  
- Ready for:
  - Exploratory Data Analysis (EDA)  
  - Feature Engineering  
  - Machine Learning  

---

## Why Synthetic Data?

- Maintains **data privacy**  
- Allows control over **feature relationships**  
- Simulates **real-world financial behavior**  

---

## Next Step

 Perform **Exploratory Data Analysis (EDA)** to understand patterns, distributions, and key drivers of loan default.

In [5]:
import pandas as pd
import numpy as np


In [6]:
np.random.seed(42)
N = 10000

# 1. BASIC PROFILE

In [7]:
age = np.random.randint(21, 60, N)

employment_type = np.random.choice(
    ['salaried', 'self-employed', 'unemployed'],
    N,
    p=[0.6, 0.3, 0.1]
)

employment_length = np.array([
    np.random.randint(0, 20) if et != 'unemployed' else 0
    for et in employment_type
])

# 2. INCOME

In [8]:
base_income = np.random.lognormal(mean=12, sigma=0.7, size=N)

annual_income = np.array([
    inc * 1.2 if et == 'self-employed' else
    inc * 0.5 if et == 'unemployed' else
    inc
    for inc, et in zip(base_income, employment_type)
])

annual_income = np.clip(annual_income, 200000, 5000000)

income_stability_score = (
    (employment_length / 20) * 50 +
    np.where(employment_type == 'salaried', 40, 20)
)

income_stability_score = np.clip(income_stability_score, 20, 100)

# 3. LOAN TYPE + TERM

In [9]:
loan_type = np.random.choice(
    ['personal', 'home', 'auto', 'education'],
    N,
    p=[0.4, 0.2, 0.2, 0.2]
)

loan_term = []
for lt in loan_type:
    if lt == 'personal':
        loan_term.append(np.random.choice([12, 24, 36, 48]))
    elif lt == 'auto':
        loan_term.append(np.random.choice([36, 48, 60]))
    elif lt == 'education':
        loan_term.append(np.random.choice([36, 60, 84]))
    else:
        loan_term.append(np.random.choice([120, 180, 240, 300]))

loan_term = np.array(loan_term)

# 4. LOAN AMOUNT 

In [10]:
loan_amount = []

for i in range(N):
    inc = annual_income[i]
    lt = loan_type[i]

    if lt == 'personal':
        amt = inc * np.random.uniform(0.2, 0.8)
    elif lt == 'auto':
        amt = inc * np.random.uniform(0.5, 1.5)
    elif lt == 'education':
        amt = inc * np.random.uniform(0.3, 1.0)
    else:
        amt = inc * np.random.uniform(2, 6)

    loan_amount.append(amt)

loan_amount = np.array(loan_amount)

# 5. CREDIT SCORE

In [11]:
credit_score = np.random.normal(680, 80, N)
credit_score = np.clip(credit_score, 300, 850)

# 6. INTEREST RATE 

In [12]:
interest_rate = []

for i in range(N):
    base = 8

    if credit_score[i] < 600:
        base += 4
    elif credit_score[i] < 700:
        base += 2

    rate = base + np.random.uniform(0, 2)
    interest_rate.append(rate)

interest_rate = np.array(interest_rate)

# 7. EMI 

In [13]:
monthly_rate = interest_rate / (12 * 100)

emi = (loan_amount * monthly_rate * (1 + monthly_rate) ** loan_term) / \
      ((1 + monthly_rate) ** loan_term - 1)

# 8. FINANCIAL FEATURES

In [14]:
total_existing_loans = np.random.poisson(2, N)

total_debt = loan_amount + (annual_income * np.random.uniform(0.1, 0.8, N))

dti = total_debt / annual_income
emi_to_income = emi / (annual_income / 12)
loan_to_income = loan_amount / annual_income

# 9. BEHAVIORAL FEATURES

In [15]:
num_inquiries = np.random.poisson(3, N)

late_payments = np.random.poisson(1.5, N)

default_history = np.where(late_payments > 3, 1, 0)

credit_utilization = np.clip(
    np.random.beta(2, 5, N),
    0.05, 0.95
)

repayment_score = np.clip(
    100 - (late_payments * 10 + credit_utilization * 50),
    20, 100
)

credit_history_length = np.clip(
    employment_length + np.random.randint(0, 5, N),
    1, 25
)

job_stability_score = np.clip(
    (employment_length / 20) * 100,
    10, 100
)

spending_to_income = np.clip(
    np.random.normal(0.5, 0.15, N),
    0.1, 0.9
)

# 10. ADVANCED RISK-BASED TARGET

In [16]:
# Normalize credit score
credit_norm = (credit_score - 300) / (850 - 300)

risk_score = (
    0.30 * (1 - credit_norm) +
    0.25 * dti +
    0.20 * emi_to_income +
    0.15 * credit_utilization +
    0.10 * (late_payments / 10)
)

# Behavioral penalties
risk_score += (default_history * 0.2)
risk_score += (num_inquiries > 5) * 0.1

# Clip
risk_score = np.clip(risk_score, 0, 1)

# Add noise (real-world uncertainty)
prob_default = risk_score + np.random.normal(0, 0.05, N)
prob_default = np.clip(prob_default, 0, 1)

# Final classification
loan_status = (prob_default > 0.5).astype(int)

# 11. FINAL DATAFRAME

In [17]:
df = pd.DataFrame({
    "user_id": range(1, N+1),
    "age": age,
    "employment_type": employment_type,
    "employment_length_years": employment_length,
    "annual_income": annual_income,
    "income_stability_score": income_stability_score,
    "loan_amount": loan_amount,
    "loan_term_months": loan_term,
    "interest_rate": interest_rate,
    "loan_type": loan_type,
    "emi": emi,
    "total_existing_loans": total_existing_loans,
    "total_debt": total_debt,
    "debt_to_income_ratio": dti,
    "credit_score": credit_score,
    "credit_utilization_ratio": credit_utilization,
    "emi_to_income_ratio": emi_to_income,
    "loan_to_income_ratio": loan_to_income,
    "num_credit_inquiries": num_inquiries,
    "late_payment_count": late_payments,
    "default_history": default_history,
    "repayment_history_score": repayment_score,
    "credit_history_length_years": credit_history_length,
    "job_stability_score": job_stability_score,
    "spending_to_income_ratio": spending_to_income,
    "loan_status": loan_status
})

# 12. SAVE

In [20]:
df.to_csv("../data/raw/creditpilot_dataset.csv", index=False)